# 🧹 Online Retail Dataset – Step 2: Data Cleaning

**Mục tiêu:** Đánh giá chất lượng dữ liệu, phát hiện và xử lý outliers, missing values, chuẩn hóa dữ liệu.

| Cell | Nội dung |
|------|----------|
| 1 | Data Quality Assessment & Outlier Detection |
| 2 | Outlier Investigation |
| 3 | Outlier Treatment Decisions |
| 4 | Missing Data Handling |
| 5 | Data Type Optimization & Standardization |
| 6 | Cleaned Data Validation & Export |

> ⚠️ **Yêu cầu:** Chạy **Step 1 trước** để có `df` và các biến `categorical_cols`, `numerical_cols` trong session.

---
## 🔍 CELL 1: Data Quality Assessment & Outlier Detection

**Mục đích:** Tổng quan chất lượng dữ liệu — phát hiện outliers bằng IQR và Z-score, visualize bằng boxplot và scatter, kiểm tra duplicates, format không nhất quán, và phân tích missing data patterns.

In [ ]:
# ============================================================
# CELL 1: DATA QUALITY ASSESSMENT & OUTLIER DETECTION
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── If running standalone, reload data ────────────────────────
try:
    _ = df
except NameError:
    from google.colab import files
    print("⚠️  df not found. Please upload your file:")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    df = pd.read_excel(fname) if fname.endswith(('.xlsx','.xls')) else pd.read_csv(fname, encoding='unicode_escape')

df_raw = df.copy()   # keep original for comparison later

# ── Column definitions ────────────────────────────────────────
categorical_cols = ['InvoiceNo', 'StockCode', 'Description', 'InvoiceDate', 'Country']
numerical_cols   = ['Quantity', 'UnitPrice', 'CustomerID']
num_analysis     = ['Quantity', 'UnitPrice']   # columns to analyse for outliers (exclude ID)

print("=" * 65)
print("📊  DATA QUALITY ASSESSMENT REPORT")
print("=" * 65)

# ── 1A. Basic shape ───────────────────────────────────────────
print(f"\n📐 Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")

# ── 1B. Missing values ────────────────────────────────────────
print("\n❓ Missing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
mv_df = pd.DataFrame({'Count': missing, '%': missing_pct})
mv_df = mv_df[mv_df['Count'] > 0].sort_values('Count', ascending=False)
print(mv_df.to_string() if not mv_df.empty else "   ✅ No missing values")

# ── 1C. Duplicates ────────────────────────────────────────────
n_dup = df.duplicated().sum()
print(f"\n🔁 Duplicate Rows : {n_dup:,} ({n_dup/len(df)*100:.2f}%)")

# ── 1D. Negative / zero values (business logic checks) ───────
print("\n⚠️  Business Logic Checks:")
neg_qty   = (df['Quantity']  < 0).sum()
zero_qty  = (df['Quantity'] == 0).sum()
neg_price = (df['UnitPrice']  < 0).sum()
zero_price= (df['UnitPrice'] == 0).sum()
print(f"   Negative Quantity  : {neg_qty:,}  (returns/cancellations)")
print(f"   Zero     Quantity  : {zero_qty:,}")
print(f"   Negative UnitPrice : {neg_price:,}")
print(f"   Zero     UnitPrice : {zero_price:,}")

# Check InvoiceNo starting with 'C' (cancellations)
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
n_cancel = df['InvoiceNo'].str.startswith('C').sum()
print(f"   Cancelled Invoices (start with 'C') : {n_cancel:,}")

# ── 1E. Inconsistent text format ─────────────────────────────
print("\n🔤 Text Format Issues:")
leading_space = df['Description'].dropna().apply(lambda x: str(x).startswith(' ')).sum()
trailing_space= df['Description'].dropna().apply(lambda x: str(x).endswith(' ')).sum()
print(f"   Leading  spaces in Description : {leading_space:,}")
print(f"   Trailing spaces in Description : {trailing_space:,}")

# ── 1F. Outlier Detection – IQR ───────────────────────────────
print("\n📦 Outlier Detection (IQR Method):")
outlier_summary = {}
for col in num_analysis:
    col_data = pd.to_numeric(df[col], errors='coerce').dropna()
    Q1, Q3   = col_data.quantile(0.25), col_data.quantile(0.75)
    IQR      = Q3 - Q1
    lb, ub   = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out    = ((col_data < lb) | (col_data > ub)).sum()
    pct_out  = n_out / len(col_data) * 100
    outlier_summary[col] = {'Q1':Q1,'Q3':Q3,'IQR':IQR,'Lower':lb,'Upper':ub,'Count':n_out,'Pct':pct_out}
    print(f"   {col:12s} → IQR bounds [{lb:.2f}, {ub:.2f}]  |  outliers: {n_out:,} ({pct_out:.1f}%)")

# ── 1G. Outlier Detection – Z-score ──────────────────────────
print("\n📊 Outlier Detection (Z-score |z|>3):")
for col in num_analysis:
    col_data = pd.to_numeric(df[col], errors='coerce').dropna()
    z        = np.abs(stats.zscore(col_data))
    n_z      = (z > 3).sum()
    print(f"   {col:12s} → {n_z:,} outliers ({n_z/len(col_data)*100:.1f}%)")

# ── 1H. Visualizations ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Data Quality Assessment – Outlier Visualization', fontsize=15, fontweight='bold')

df_num = df.copy()
for c in num_analysis:
    df_num[c] = pd.to_numeric(df_num[c], errors='coerce')

# Boxplot – Quantity
axes[0,0].boxplot(df_num['Quantity'].dropna(), patch_artist=True,
                  boxprops=dict(facecolor='#4FC3F7', color='navy'),
                  medianprops=dict(color='red', linewidth=2))
axes[0,0].set_title('Boxplot: Quantity (all)', fontweight='bold')
axes[0,0].set_ylabel('Quantity')
axes[0,0].grid(axis='y', alpha=0.3)

# Boxplot – UnitPrice
axes[0,1].boxplot(df_num['UnitPrice'].dropna(), patch_artist=True,
                  boxprops=dict(facecolor='#A5D6A7', color='darkgreen'),
                  medianprops=dict(color='red', linewidth=2))
axes[0,1].set_title('Boxplot: UnitPrice (all)', fontweight='bold')
axes[0,1].set_ylabel('UnitPrice')
axes[0,1].grid(axis='y', alpha=0.3)

# Boxplot – Quantity (positive only, zoomed)
pos_qty = df_num[df_num['Quantity'] > 0]['Quantity'].dropna()
axes[0,2].boxplot(pos_qty, patch_artist=True,
                  boxprops=dict(facecolor='#FFB74D', color='darkorange'),
                  medianprops=dict(color='red', linewidth=2))
axes[0,2].set_title('Boxplot: Quantity (positive only)', fontweight='bold')
axes[0,2].set_ylabel('Quantity')
axes[0,2].grid(axis='y', alpha=0.3)

# Scatter – Quantity vs UnitPrice
sample = df_num[['Quantity','UnitPrice']].dropna().sample(min(5000, len(df_num)), random_state=42)
axes[1,0].scatter(sample['Quantity'], sample['UnitPrice'],
                  alpha=0.3, s=8, color='steelblue')
iq = outlier_summary
axes[1,0].axvline(iq['Quantity']['Upper'], color='red',   linestyle='--', lw=1.2, label='IQR upper')
axes[1,0].axvline(iq['Quantity']['Lower'], color='orange',linestyle='--', lw=1.2, label='IQR lower')
axes[1,0].axhline(iq['UnitPrice']['Upper'],color='red',   linestyle=':', lw=1.2)
axes[1,0].set_title('Scatter: Quantity vs UnitPrice', fontweight='bold')
axes[1,0].set_xlabel('Quantity'); axes[1,0].set_ylabel('UnitPrice')
axes[1,0].legend(fontsize=8); axes[1,0].grid(alpha=0.3)

# Histogram – Quantity distribution
pos_qty_plot = df_num[(df_num['Quantity']>0) & (df_num['Quantity']<200)]['Quantity'].dropna()
axes[1,1].hist(pos_qty_plot, bins=50, color='#4FC3F7', edgecolor='navy', alpha=0.8)
axes[1,1].set_title('Histogram: Quantity (0–200)', fontweight='bold')
axes[1,1].set_xlabel('Quantity'); axes[1,1].set_ylabel('Frequency')
axes[1,1].grid(axis='y', alpha=0.3)

# Missing value heatmap (sample)
missing_flag = df.isnull().sample(min(300, len(df)), random_state=42)
sns.heatmap(missing_flag, cbar=False, yticklabels=False,
            cmap=['#E8F5E9','#EF5350'], ax=axes[1,2])
axes[1,2].set_title('Missing Value Pattern (sample 300 rows)', fontweight='bold')

plt.tight_layout()
plt.savefig('quality_assessment.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n✅ Cell 1 complete. Charts saved to quality_assessment.png")

---
## 🔬 CELL 2: Outlier Investigation

**Mục đích:** Hiển thị các record outlier để review nghiệp vụ, phân tích thống kê mức độ cực đoan, nhận diện patterns và document để ra quyết định xử lý.

In [ ]:
# ============================================================
# CELL 2: OUTLIER INVESTIGATION
# ============================================================
from scipy import stats

df['Quantity']  = pd.to_numeric(df['Quantity'],  errors='coerce')
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

# ── Helper: IQR bounds ────────────────────────────────────────
def iqr_bounds(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5*IQR, Q3 + 1.5*IQR

qty_lb,  qty_ub  = iqr_bounds(df['Quantity'].dropna())
prc_lb,  prc_ub  = iqr_bounds(df['UnitPrice'].dropna())

# ── 2A. Cancelled Invoices (InvoiceNo starts with 'C') ───────
df_cancel = df[df['InvoiceNo'].str.startswith('C')]
print("=" * 65)
print("🔴 CANCELLED INVOICES (InvoiceNo starts with 'C')")
print("=" * 65)
print(f"   Count : {len(df_cancel):,}")
print(f"   Quantity range  : {df_cancel['Quantity'].min()} to {df_cancel['Quantity'].max()}")
print(f"   UnitPrice range : {df_cancel['UnitPrice'].min()} to {df_cancel['UnitPrice'].max()}")
print("\n   Sample records:")
display(df_cancel[['InvoiceNo','StockCode','Description','Quantity','UnitPrice','Country']].head(5))

# ── 2B. High Quantity outliers ───────────────────────────────
df_high_qty = df[(df['Quantity'] > qty_ub) & (~df['InvoiceNo'].str.startswith('C'))]
print(f"\n{'='*65}")
print(f"📦 HIGH QUANTITY OUTLIERS (> IQR upper = {qty_ub:.0f})")
print(f"{'='*65}")
print(f"   Count : {len(df_high_qty):,}")
if len(df_high_qty) > 0:
    print(f"   Max Quantity : {df_high_qty['Quantity'].max():,}")
    print(f"   Countries    : {df_high_qty['Country'].value_counts().head(3).to_dict()}")
    print("\n   Top 10 extreme quantity records:")
    display(df_high_qty.nlargest(10, 'Quantity')[['InvoiceNo','StockCode','Description','Quantity','UnitPrice','Country']])

# ── 2C. High UnitPrice outliers ──────────────────────────────
df_high_prc = df[(df['UnitPrice'] > prc_ub) & (~df['InvoiceNo'].str.startswith('C'))]
print(f"\n{'='*65}")
print(f"💰 HIGH UNIT PRICE OUTLIERS (> IQR upper = {prc_ub:.2f})")
print(f"{'='*65}")
print(f"   Count : {len(df_high_prc):,}")
if len(df_high_prc) > 0:
    print(f"   Max UnitPrice : {df_high_prc['UnitPrice'].max():,}")
    print("\n   Top 10 extreme price records:")
    display(df_high_prc.nlargest(10, 'UnitPrice')[['InvoiceNo','StockCode','Description','Quantity','UnitPrice','Country']])

# ── 2D. Negative Quantity (returns) ──────────────────────────
df_neg = df[df['Quantity'] < 0]
print(f"\n{'='*65}")
print("🔻 NEGATIVE QUANTITY (Returns / Adjustments)")
print(f"{'='*65}")
print(f"   Count  : {len(df_neg):,}")
if len(df_neg) > 0:
    print(f"   Min    : {df_neg['Quantity'].min():,}")
    print(f"   Linked to cancelled InvoiceNo: {df_neg['InvoiceNo'].str.startswith('C').sum():,}")
    print("\n   Sample:")
    display(df_neg[['InvoiceNo','Description','Quantity','UnitPrice','Country']].head(5))

# ── 2E. Zero UnitPrice ───────────────────────────────────────
df_zero_prc = df[df['UnitPrice'] == 0]
print(f"\n{'='*65}")
print("🆓 ZERO UNIT PRICE RECORDS (Free samples / Data errors)")
print(f"{'='*65}")
print(f"   Count : {len(df_zero_prc):,}")
if len(df_zero_prc) > 0:
    display(df_zero_prc[['InvoiceNo','StockCode','Description','Quantity','Country']].head(5))

# ── 2F. Z-score statistical significance ─────────────────────
print(f"\n{'='*65}")
print("📊 Z-SCORE ANALYSIS (statistical significance)")
print(f"{'='*65}")
for col in ['Quantity','UnitPrice']:
    pos_data = df[df[col] > 0][col].dropna()
    z = np.abs(stats.zscore(pos_data))
    extreme = pos_data[z > 5]
    print(f"   {col}: {(z>3).sum():,} records |z|>3  |  {(z>5).sum():,} records |z|>5")
    if len(extreme) > 0:
        print(f"      Extreme (|z|>5) range: {extreme.min():.2f} – {extreme.max():.2f}")

print("\n✅ Cell 2 complete – review outlier records above before Cell 3.")

---
## ⚖️ CELL 3: Outlier Treatment Decisions

**Mục đích:** Cung cấp nhiều lựa chọn xử lý cho từng loại outlier (remove / keep / cap), áp dụng business-logic phù hợp với dữ liệu retail, và ghi lại toàn bộ quyết định.

In [ ]:
# ============================================================
# CELL 3: OUTLIER TREATMENT DECISIONS
# ============================================================

# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIGURE YOUR CHOICES HERE  (edit True/False/values)  ║
# ╚══════════════════════════════════════════════════════════╝

REMOVE_CANCELLED_INVOICES  = True    # Remove rows where InvoiceNo starts with 'C'
REMOVE_NEGATIVE_QUANTITY   = True    # Remove rows with Quantity < 0 (returns)
REMOVE_ZERO_UNITPRICE      = True    # Remove rows with UnitPrice == 0

# Quantity outlier strategy: 'cap' | 'remove' | 'keep'
QUANTITY_OUTLIER_STRATEGY  = 'cap'
QUANTITY_CAP_PERCENTILE    = 99      # cap at this percentile (if strategy='cap')

# UnitPrice outlier strategy: 'cap' | 'remove' | 'keep'
UNITPRICE_OUTLIER_STRATEGY = 'cap'
UNITPRICE_CAP_PERCENTILE   = 99

# ─────────────────────────────────────────────────────────────

cleaning_log = []
df_clean = df.copy()
n_before = len(df_clean)

print("=" * 65)
print("⚖️   OUTLIER TREATMENT DECISIONS")
print("=" * 65)

# ── Decision 1: Cancelled Invoices ────────────────────────────
print("\n📌 Decision 1 – Cancelled Invoices (InvoiceNo starts 'C')")
print("   Options:")
print("   [A] REMOVE  → drop rows (recommended for sales analysis)")
print("   [B] KEEP    → keep for return/refund analysis")
if REMOVE_CANCELLED_INVOICES:
    n = df_clean['InvoiceNo'].str.startswith('C').sum()
    df_clean = df_clean[~df_clean['InvoiceNo'].str.startswith('C')]
    cleaning_log.append({'Step':'Remove cancelled invoices','Rows_removed':n,'Strategy':'Remove (InvoiceNo starts C)'})
    print(f"   ✅ CHOSEN: REMOVE → {n:,} rows removed")
else:
    cleaning_log.append({'Step':'Cancelled invoices','Rows_removed':0,'Strategy':'Kept'})
    print("   ✅ CHOSEN: KEEP")

# ── Decision 2: Negative Quantity ────────────────────────────
print("\n📌 Decision 2 – Negative Quantity (returns / credit notes)")
print("   Options:")
print("   [A] REMOVE  → for pure sales analysis")
print("   [B] KEEP    → for return rate / churn analysis")
if REMOVE_NEGATIVE_QUANTITY:
    n = (df_clean['Quantity'] < 0).sum()
    df_clean = df_clean[df_clean['Quantity'] >= 0]
    cleaning_log.append({'Step':'Remove negative Quantity','Rows_removed':n,'Strategy':'Remove (Qty<0)'})
    print(f"   ✅ CHOSEN: REMOVE → {n:,} rows removed")
else:
    cleaning_log.append({'Step':'Negative Quantity','Rows_removed':0,'Strategy':'Kept'})
    print("   ✅ CHOSEN: KEEP")

# ── Decision 3: Zero UnitPrice ────────────────────────────────
print("\n📌 Decision 3 – Zero UnitPrice (free samples / errors)")
print("   Options:")
print("   [A] REMOVE  → likely data errors or gifts")
print("   [B] KEEP    → if free samples are relevant")
if REMOVE_ZERO_UNITPRICE:
    n = (df_clean['UnitPrice'] == 0).sum()
    df_clean = df_clean[df_clean['UnitPrice'] > 0]
    cleaning_log.append({'Step':'Remove zero UnitPrice','Rows_removed':n,'Strategy':'Remove (Price=0)'})
    print(f"   ✅ CHOSEN: REMOVE → {n:,} rows removed")
else:
    cleaning_log.append({'Step':'Zero UnitPrice','Rows_removed':0,'Strategy':'Kept'})
    print("   ✅ CHOSEN: KEEP")

# ── Decision 4: Quantity outliers (extreme high) ──────────────
print(f"\n📌 Decision 4 – Extreme HIGH Quantity outliers")
print("   Options:")
print("   [A] CAP     → replace extreme values with percentile cap")
print("   [B] REMOVE  → drop extreme rows")
print("   [C] KEEP    → keep as-is (bulk orders are real)")

if QUANTITY_OUTLIER_STRATEGY == 'cap':
    cap_val = df_clean['Quantity'].quantile(QUANTITY_CAP_PERCENTILE/100)
    n = (df_clean['Quantity'] > cap_val).sum()
    df_clean['Quantity'] = df_clean['Quantity'].clip(upper=cap_val)
    cleaning_log.append({'Step':f'Cap Quantity at p{QUANTITY_CAP_PERCENTILE}','Rows_removed':0,'Strategy':f'Cap at {cap_val:.0f}  ({n} values capped)'})
    print(f"   ✅ CHOSEN: CAP at p{QUANTITY_CAP_PERCENTILE} ({cap_val:.0f}) → {n:,} values capped")
elif QUANTITY_OUTLIER_STRATEGY == 'remove':
    Q1, Q3 = df_clean['Quantity'].quantile([0.25,0.75])
    ub = Q3 + 1.5*(Q3-Q1)
    n = (df_clean['Quantity'] > ub).sum()
    df_clean = df_clean[df_clean['Quantity'] <= ub]
    cleaning_log.append({'Step':'Remove Quantity IQR outliers','Rows_removed':n,'Strategy':f'Remove > IQR upper ({ub:.0f})'})
    print(f"   ✅ CHOSEN: REMOVE IQR outliers → {n:,} rows removed")
else:
    cleaning_log.append({'Step':'Quantity outliers','Rows_removed':0,'Strategy':'Kept'})
    print("   ✅ CHOSEN: KEEP – bulk orders treated as legitimate")

# ── Decision 5: UnitPrice outliers ───────────────────────────
print(f"\n📌 Decision 5 – Extreme HIGH UnitPrice outliers")
print("   Options:")
print("   [A] CAP     → replace extreme values with percentile cap")
print("   [B] REMOVE  → drop extreme rows")
print("   [C] KEEP    → luxury/premium items may have high price")

if UNITPRICE_OUTLIER_STRATEGY == 'cap':
    cap_val = df_clean['UnitPrice'].quantile(UNITPRICE_CAP_PERCENTILE/100)
    n = (df_clean['UnitPrice'] > cap_val).sum()
    df_clean['UnitPrice'] = df_clean['UnitPrice'].clip(upper=cap_val)
    cleaning_log.append({'Step':f'Cap UnitPrice at p{UNITPRICE_CAP_PERCENTILE}','Rows_removed':0,'Strategy':f'Cap at {cap_val:.2f}  ({n} values capped)'})
    print(f"   ✅ CHOSEN: CAP at p{UNITPRICE_CAP_PERCENTILE} ({cap_val:.2f}) → {n:,} values capped")
elif UNITPRICE_OUTLIER_STRATEGY == 'remove':
    Q1, Q3 = df_clean['UnitPrice'].quantile([0.25,0.75])
    ub = Q3 + 1.5*(Q3-Q1)
    n = (df_clean['UnitPrice'] > ub).sum()
    df_clean = df_clean[df_clean['UnitPrice'] <= ub]
    cleaning_log.append({'Step':'Remove UnitPrice IQR outliers','Rows_removed':n,'Strategy':f'Remove > IQR upper ({ub:.2f})'})
    print(f"   ✅ CHOSEN: REMOVE IQR outliers → {n:,} rows removed")
else:
    cleaning_log.append({'Step':'UnitPrice outliers','Rows_removed':0,'Strategy':'Kept'})
    print("   ✅ CHOSEN: KEEP")

# ── Summary table ─────────────────────────────────────────────
print(f"\n{'='*65}")
print("📋 OUTLIER TREATMENT SUMMARY")
print(f"{'='*65}")
log_df = pd.DataFrame(cleaning_log)
display(log_df)
print(f"\n   Rows before outlier treatment : {n_before:,}")
print(f"   Rows after  outlier treatment : {len(df_clean):,}")
print(f"   Total rows removed            : {n_before - len(df_clean):,}")
print("\n✅ Cell 3 complete.")

---
## 🩹 CELL 4: Missing Data Handling

**Mục đích:** Xử lý tự động missing values theo loại cột — imputation median cho số, mode cho categorical — và ghi lại toàn bộ quyết định.

In [ ]:
# ============================================================
# CELL 4: MISSING DATA HANDLING
# ============================================================

missing_log = []
n_before_mv = len(df_clean)

print("=" * 65)
print("🩹  MISSING DATA HANDLING")
print("=" * 65)

# ── 4A. Missing counts before ────────────────────────────────
print("\n📊 Missing Values BEFORE treatment:")
mv_before = df_clean.isnull().sum()
mv_before = mv_before[mv_before > 0]
if mv_before.empty:
    print("   ✅ No missing values remaining in cleaned dataset!")
else:
    for col, cnt in mv_before.items():
        pct = cnt / len(df_clean) * 100
        print(f"   {col:15s}: {cnt:,} ({pct:.1f}%)")

# ── 4B. CustomerID – high missing (expected in retail) ───────
print("\n📌 Strategy: CustomerID (Guest / Anonymous transactions)")
n_miss_cid = df_clean['CustomerID'].isnull().sum()
print(f"   Missing count : {n_miss_cid:,}")
print("   Options:")
print("   [A] FILL with 0  → mark as 'Guest' customer")
print("   [B] DROP rows    → keep only identified customers")
print("   [C] FILL -1      → sentinel for 'unknown'")

# ── CONFIGURE: change strategy here ──────────────────────────
CUSTOMERID_STRATEGY = 'fill_zero'   # 'fill_zero' | 'drop' | 'fill_neg1'

if CUSTOMERID_STRATEGY == 'fill_zero':
    df_clean['CustomerID'] = df_clean['CustomerID'].fillna(0).astype(int)
    missing_log.append({'Column':'CustomerID','Strategy':'Fill 0 (Guest)','Filled':n_miss_cid,'Dropped':0})
    print(f"   ✅ CHOSEN: Fill with 0 → {n_miss_cid:,} values filled")
elif CUSTOMERID_STRATEGY == 'drop':
    df_clean = df_clean[df_clean['CustomerID'].notna()]
    missing_log.append({'Column':'CustomerID','Strategy':'Drop rows','Filled':0,'Dropped':n_miss_cid})
    print(f"   ✅ CHOSEN: Drop rows → {n_miss_cid:,} rows removed")
else:
    df_clean['CustomerID'] = df_clean['CustomerID'].fillna(-1).astype(int)
    missing_log.append({'Column':'CustomerID','Strategy':'Fill -1 (Unknown)','Filled':n_miss_cid,'Dropped':0})
    print(f"   ✅ CHOSEN: Fill with -1 → {n_miss_cid:,} values filled")

# ── 4C. Description – categorical, fill with 'Unknown' ───────
print("\n📌 Strategy: Description (product text)")
n_miss_desc = df_clean['Description'].isnull().sum()
print(f"   Missing count : {n_miss_desc:,}")
if n_miss_desc > 0:
    df_clean['Description'] = df_clean['Description'].fillna('Unknown')
    missing_log.append({'Column':'Description','Strategy':'Fill Unknown','Filled':n_miss_desc,'Dropped':0})
    print(f"   ✅ Filled with 'Unknown' → {n_miss_desc:,} values")
else:
    print("   ✅ No missing values.")

# ── 4D. Numerical columns – fill with median ─────────────────
print("\n📌 Strategy: Numerical columns (Quantity, UnitPrice) → median imputation")
for col in ['Quantity','UnitPrice']:
    n_miss = df_clean[col].isnull().sum()
    if n_miss > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        missing_log.append({'Column':col,'Strategy':f'Fill median ({median_val:.2f})','Filled':n_miss,'Dropped':0})
        print(f"   {col}: filled {n_miss:,} with median = {median_val:.2f}")
    else:
        print(f"   {col}: ✅ No missing values.")

# ── 4E. Remaining columns – drop if still missing ────────────
remaining_miss = df_clean.isnull().sum()
remaining_miss = remaining_miss[remaining_miss > 0]
if not remaining_miss.empty:
    print("\n📌 Remaining missing columns → drop rows:")
    for col, cnt in remaining_miss.items():
        df_clean = df_clean[df_clean[col].notna()]
        missing_log.append({'Column':col,'Strategy':'Drop rows','Filled':0,'Dropped':cnt})
        print(f"   {col}: dropped {cnt:,} rows")

# ── 4F. Summary ───────────────────────────────────────────────
print(f"\n{'='*65}")
print("📋 MISSING DATA TREATMENT SUMMARY")
print(f"{'='*65}")
if missing_log:
    display(pd.DataFrame(missing_log))
print(f"\n   Rows before : {n_before_mv:,}")
print(f"   Rows after  : {len(df_clean):,}")
print(f"   Rows lost   : {n_before_mv - len(df_clean):,}")
print(f"\n   Missing values remaining : {df_clean.isnull().sum().sum()}")
print("\n✅ Cell 4 complete.")

---
## 🔧 CELL 5: Data Type Optimization & Standardization

**Mục đích:** Chuyển đổi kiểu dữ liệu chuẩn, parse InvoiceDate, loại bỏ duplicates, chuẩn hóa text (trim, title-case), tạo cột Revenue, và normalize/standardize numerical columns.

In [ ]:
# ============================================================
# CELL 5: DATA TYPE OPTIMIZATION & STANDARDIZATION
# ============================================================

std_log = []

print("=" * 65)
print("🔧  DATA TYPE OPTIMIZATION & STANDARDIZATION")
print("=" * 65)

# ── 5A. Remove duplicates ─────────────────────────────────────
n_before_dup = len(df_clean)
df_clean = df_clean.drop_duplicates()
n_dup_removed = n_before_dup - len(df_clean)
std_log.append({'Step':'Remove duplicates','Detail':f'{n_dup_removed:,} rows removed'})
print(f"\n♻️  Duplicates removed : {n_dup_removed:,}")

# ── 5B. Parse InvoiceDate → datetime ─────────────────────────
print("\n📅 Parsing InvoiceDate to datetime...")
df_clean['InvoiceDate'] = df_clean['InvoiceDate'].astype(str)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], infer_datetime_format=True, errors='coerce')
n_bad_dates = df_clean['InvoiceDate'].isnull().sum()
df_clean = df_clean[df_clean['InvoiceDate'].notna()]

# Extract temporal features
df_clean['Year']    = df_clean['InvoiceDate'].dt.year
df_clean['Month']   = df_clean['InvoiceDate'].dt.month
df_clean['Day']     = df_clean['InvoiceDate'].dt.day
df_clean['Weekday'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour']    = df_clean['InvoiceDate'].dt.hour

std_log.append({'Step':'Parse InvoiceDate','Detail':f'Datetime parsed; {n_bad_dates} unparseable dropped; Year/Month/Day/Weekday/Hour extracted'})
print(f"   ✅ Datetime parsed. Unparseable dropped: {n_bad_dates}")
print(f"   Date range: {df_clean['InvoiceDate'].min().date()} → {df_clean['InvoiceDate'].max().date()}")

# ── 5C. Text standardization ─────────────────────────────────
print("\n🔤 Standardizing text columns...")

# Description: strip whitespace + title case
df_clean['Description'] = df_clean['Description'].astype(str).str.strip().str.title()
std_log.append({'Step':'Description cleanup','Detail':'strip() + title()'})
print("   ✅ Description: stripped whitespace, applied title case")

# Country: strip + title case
df_clean['Country'] = df_clean['Country'].astype(str).str.strip().str.title()
std_log.append({'Step':'Country cleanup','Detail':'strip() + title()'})
print("   ✅ Country: stripped whitespace, applied title case")

# InvoiceNo: uppercase + strip
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype(str).str.strip().str.upper()
std_log.append({'Step':'InvoiceNo cleanup','Detail':'strip() + upper()'})
print("   ✅ InvoiceNo: stripped + uppercased")

# StockCode: strip + upper
df_clean['StockCode'] = df_clean['StockCode'].astype(str).str.strip().str.upper()
std_log.append({'Step':'StockCode cleanup','Detail':'strip() + upper()'})
print("   ✅ StockCode: stripped + uppercased")

# ── 5D. Create Revenue column ─────────────────────────────────
df_clean['Revenue'] = (df_clean['Quantity'] * df_clean['UnitPrice']).round(2)
std_log.append({'Step':'Revenue column','Detail':'Quantity × UnitPrice'})
print("\n💰 Revenue column created (Quantity × UnitPrice)")

# ── 5E. Optimize dtypes ───────────────────────────────────────
print("\n🗜️  Optimizing dtypes...")
df_clean['Quantity']   = df_clean['Quantity'].astype('float32')
df_clean['UnitPrice']  = df_clean['UnitPrice'].astype('float32')
df_clean['Revenue']    = df_clean['Revenue'].astype('float32')
df_clean['CustomerID'] = df_clean['CustomerID'].astype('int32')
df_clean['Year']       = df_clean['Year'].astype('int16')
df_clean['Month']      = df_clean['Month'].astype('int8')
df_clean['Day']        = df_clean['Day'].astype('int8')
df_clean['Hour']       = df_clean['Hour'].astype('int8')
for col in ['Country','Weekday','StockCode']:
    df_clean[col] = df_clean[col].astype('category')
std_log.append({'Step':'Dtype optimization','Detail':'float32/int32/int16/int8/category applied'})
print("   ✅ Numeric downcasted to float32/int32; Country/Weekday/StockCode → category")

# ── 5F. Normalize Revenue (Min-Max) ──────────────────────────
print("\n📐 Normalizing Revenue (Min-Max to [0,1])...")
rev_min = df_clean['Revenue'].min()
rev_max = df_clean['Revenue'].max()
df_clean['Revenue_Normalized'] = ((df_clean['Revenue'] - rev_min) / (rev_max - rev_min)).astype('float32')

# Standardize Quantity (Z-score)
print("📐 Standardizing Quantity (Z-score)...")
qty_mean = df_clean['Quantity'].mean()
qty_std  = df_clean['Quantity'].std()
df_clean['Quantity_ZScore'] = ((df_clean['Quantity'] - qty_mean) / qty_std).astype('float32')
std_log.append({'Step':'Normalization','Detail':'Revenue: Min-Max [0,1]; Quantity: Z-score'})
print("   ✅ Revenue_Normalized and Quantity_ZScore columns added")

# ── Summary ───────────────────────────────────────────────────
print(f"\n{'='*65}")
print("📋 STANDARDIZATION SUMMARY")
print(f"{'='*65}")
display(pd.DataFrame(std_log))
print(f"\n   Final shape : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print("\n✅ Cell 5 complete.")

---
## ✅ CELL 6: Cleaned Data Validation & Export

**Mục đích:** So sánh before vs after, validate kết quả cleaning, xuất báo cáo chất lượng cuối cùng, và export cleaned dataset.

In [ ]:
# ============================================================
# CELL 6: CLEANED DATA VALIDATION & EXPORT
# ============================================================

print("=" * 65)
print("✅  CLEANED DATA VALIDATION")
print("=" * 65)

# ── 6A. Before vs After comparison ───────────────────────────
print("\n📊 BEFORE vs AFTER:")
comparison = pd.DataFrame({
    'Metric'    : ['Total Rows','Total Columns','Missing Values','Duplicate Rows',
                   'Negative Quantity','Zero UnitPrice','Cancelled Invoices'],
    'BEFORE'    : [
        len(df_raw),
        df_raw.shape[1],
        df_raw.isnull().sum().sum(),
        df_raw.duplicated().sum(),
        (pd.to_numeric(df_raw['Quantity'], errors='coerce') < 0).sum(),
        (pd.to_numeric(df_raw['UnitPrice'],errors='coerce') == 0).sum(),
        df_raw['InvoiceNo'].astype(str).str.startswith('C').sum()
    ],
    'AFTER'     : [
        len(df_clean),
        df_clean.shape[1],
        df_clean.isnull().sum().sum(),
        df_clean.duplicated().sum(),
        (df_clean['Quantity'] < 0).sum(),
        (df_clean['UnitPrice'] == 0).sum(),
        df_clean['InvoiceNo'].astype(str).str.startswith('C').sum()
    ]
})
comparison['Change'] = comparison['AFTER'] - comparison['BEFORE']
display(comparison)

# ── 6B. Final data quality checks ────────────────────────────
print(f"\n{'='*65}")
print("🔎 VALIDATION CHECKS")
print(f"{'='*65}")
checks = [
    ('No missing values',        df_clean.isnull().sum().sum() == 0),
    ('No duplicates',            df_clean.duplicated().sum() == 0),
    ('Quantity all non-negative', (df_clean['Quantity'] >= 0).all()),
    ('UnitPrice all positive',   (df_clean['UnitPrice'] > 0).all()),
    ('Revenue column exists',    'Revenue' in df_clean.columns),
    ('InvoiceDate is datetime',  pd.api.types.is_datetime64_any_dtype(df_clean['InvoiceDate'])),
    ('No cancelled invoices',    not df_clean['InvoiceNo'].str.startswith('C').any()),
]
all_pass = True
for check, result in checks:
    icon = '✅' if result else '❌'
    if not result: all_pass = False
    print(f"   {icon}  {check}")

print(f"\n   {'🎉 ALL CHECKS PASSED' if all_pass else '⚠️  SOME CHECKS FAILED – review above'}")

# ── 6C. Descriptive stats comparison ─────────────────────────
print(f"\n{'='*65}")
print("📈 NUMERICAL STATS – After Cleaning")
print(f"{'='*65}")
display(df_clean[['Quantity','UnitPrice','Revenue']].describe().round(2))

# ── 6D. Final dtype overview ──────────────────────────────────
print(f"\n{'='*65}")
print("🔠 FINAL DATA TYPES")
print(f"{'='*65}")
print(df_clean.dtypes.to_string())

# ── 6E. Visualization – Before/After boxplots ────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Before vs After Cleaning – Distribution Comparison', fontsize=14, fontweight='bold')

raw_qty = pd.to_numeric(df_raw['Quantity'],  errors='coerce').dropna()
raw_prc = pd.to_numeric(df_raw['UnitPrice'], errors='coerce').dropna()

axes[0,0].boxplot(raw_qty, patch_artist=True, boxprops=dict(facecolor='#EF9A9A'))
axes[0,0].set_title('Quantity – BEFORE', fontweight='bold'); axes[0,0].set_ylabel('Quantity'); axes[0,0].grid(axis='y',alpha=0.3)

axes[0,1].boxplot(df_clean['Quantity'].dropna(), patch_artist=True, boxprops=dict(facecolor='#A5D6A7'))
axes[0,1].set_title('Quantity – AFTER', fontweight='bold'); axes[0,1].set_ylabel('Quantity'); axes[0,1].grid(axis='y',alpha=0.3)

axes[1,0].boxplot(raw_prc, patch_artist=True, boxprops=dict(facecolor='#EF9A9A'))
axes[1,0].set_title('UnitPrice – BEFORE', fontweight='bold'); axes[1,0].set_ylabel('UnitPrice'); axes[1,0].grid(axis='y',alpha=0.3)

axes[1,1].boxplot(df_clean['UnitPrice'].dropna(), patch_artist=True, boxprops=dict(facecolor='#A5D6A7'))
axes[1,1].set_title('UnitPrice – AFTER', fontweight='bold'); axes[1,1].set_ylabel('UnitPrice'); axes[1,1].grid(axis='y',alpha=0.3)

plt.tight_layout()
plt.savefig('before_after_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 6F. Revenue distribution ─────────────────────────────────
fig2, ax = plt.subplots(figsize=(10,4))
sample_rev = df_clean[df_clean['Revenue'] < df_clean['Revenue'].quantile(0.99)]['Revenue']
ax.hist(sample_rev, bins=60, color='#42A5F5', edgecolor='navy', alpha=0.8)
ax.set_title('Revenue Distribution (p99 cap for viz)', fontweight='bold')
ax.set_xlabel('Revenue (£)'); ax.set_ylabel('Frequency'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('revenue_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 6G. Export cleaned dataset ───────────────────────────────
print(f"\n{'='*65}")
print("💾 EXPORT CLEANED DATASET")
print(f"{'='*65}")

# Export to CSV
output_csv = 'online_retail_cleaned.csv'
df_clean.to_csv(output_csv, index=False)
print(f"   ✅ CSV exported : {output_csv}")

# Export to Excel (optional)
output_xlsx = 'online_retail_cleaned.xlsx'
df_clean.to_excel(output_xlsx, index=False, engine='openpyxl')
print(f"   ✅ Excel exported : {output_xlsx}")

# Download files
from google.colab import files
print("\n📥 Downloading cleaned files...")
files.download(output_csv)
files.download(output_xlsx)

print(f"\n{'='*65}")
print(f"🎉  STEP 2 COMPLETE!")
print(f"{'='*65}")
print(f"   Raw rows      : {len(df_raw):,}")
print(f"   Cleaned rows  : {len(df_clean):,}")
print(f"   Retention     : {len(df_clean)/len(df_raw)*100:.1f}%")
print(f"   Final columns : {list(df_clean.columns)}")
print(f"\n   ➡️  df_clean is ready for Step 3: EDA & Analysis!")